In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import json

In [ ]:
# Same seeds as training so the trajectory split is identical for every config
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
data = np.load(r'C:\Users\kteja\Downloads\learning-physical-dynamics-across-resolutions-group-12\learning-physical-dynamics-across-resolutions-group-12\trainv')
X = data['field'][:, None, :].astype(np.float32)
Y = data['target'][:, None, :].astype(np.float32)
print("Data:", X.shape, Y.shape)

In [ ]:
# Trajectory-level split: rows t*5..t*5+4 form one trajectory, so row-level splitting would leak nearby time steps into validation.
n_traj = 48
perm = np.random.permutation(n_traj)
val_traj = perm[:10]
train_traj = perm[10:]
val_idx = np.concatenate([np.arange(t*5, t*5+5) for t in val_traj])
train_idx = np.concatenate([np.arange(t*5, t*5+5) for t in train_traj])

In [ ]:
X_train = torch.tensor(X[train_idx])
Y_train = torch.tensor(Y[train_idx])
X_val   = torch.tensor(X[val_idx])
Y_val   = torch.tensor(Y[val_idx])
print(f"Train: {len(X_train)}  Val: {len(X_val)}")

In [ ]:
class GridLockedCNN(nn.Module):
    def __init__(self, hidden=64, kernel_size=5, num_layers=4):
        super().__init__()
        layers = [nn.Conv1d(1, hidden, kernel_size,
                            padding=kernel_size // 2,
                            padding_mode='circular'),
                  nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Conv1d(hidden, hidden, kernel_size,
                                 padding=kernel_size // 2,
                                 padding_mode='circular'),
                       nn.ReLU()]
        layers.append(nn.Conv1d(hidden, 1, kernel_size,
                                padding=kernel_size // 2,
                                padding_mode='circular'))
        self.net = nn.Sequential(*layers)

    def forward(self, u):
        return u + self.net(u)

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(pred - target, dim=(1, 2))
    den = torch.norm(target, dim=(1, 2)) + 1e-8
    return (num / den).mean()

In [ ]:
def train_config(hidden, kernel_size, num_layers, max_epochs=400, patience=40, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = GridLockedCNN(hidden, kernel_size, num_layers)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=32, shuffle=True)

    best_val = float('inf')
    wait = 0
    lr_wait = 0
    best_state = None
    history = {'train': [], 'val': []}

    for epoch in range(max_epochs):
        # train
        model.train()
        total = 0.0
        for xb, yb in loader:
            loss = relative_l2(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        train_loss = total / len(X_train)

        # validate
        model.eval()
        with torch.no_grad():
            val_loss = relative_l2(model(X_val), Y_val).item()

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        # callbacks
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            wait = 0
            lr_wait = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            lr_wait += 1

        if wait >= patience:
            break
        if lr_wait >= 20:
            for g in optimizer.param_groups:
                g['lr'] *= 0.5
            lr_wait = 0

    n_params = sum(p.numel() for p in model.parameters())
    return best_val, n_params, history

In [ ]:
# 8 configs covering the three main architecture hyperparameters:
#   depth  : 2, 4, 6, 8
#   kernel : 3, 5, 7
#   width  : 32, 64, 128
# Standard ablation range for a small CNN
configs = [
    {'hidden': 64,  'kernel_size': 5, 'num_layers': 2},
    {'hidden': 64,  'kernel_size': 5, 'num_layers': 4},   # baseline
    {'hidden': 64,  'kernel_size': 5, 'num_layers': 6},
    {'hidden': 64,  'kernel_size': 5, 'num_layers': 8},
    {'hidden': 64,  'kernel_size': 3, 'num_layers': 4},
    {'hidden': 64,  'kernel_size': 7, 'num_layers': 4},
    {'hidden': 32,  'kernel_size': 5, 'num_layers': 4},
    {'hidden': 128, 'kernel_size': 5, 'num_layers': 4},
]

In [ ]:
results = []
for cfg in configs:
    print(f"\n--- {cfg} ---")
    val, params, _ = train_config(**cfg)
    results.append({**cfg, 'val': val, 'params': params})
    print(f"  best val = {val:.6f}   params = {params:,}")

In [ ]:
print("\n=== ABLATION SUMMARY ===")
print(f"{'layers':>6} {'kernel':>7} {'hidden':>7} {'params':>10} {'best val':>12}")
for r in sorted(results, key=lambda r: r['val']):
    print(f"{r['num_layers']:>6} {r['kernel_size']:>7} {r['hidden']:>7} "
          f"{r['params']:>10,} {r['val']:>12.6f}")

In [ ]:
with open('ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved ablation_results.json")

In [ ]:
# Selection rule: pick by validation loss ONLY. Choosing by eval-256/512 error would tune to the test metric and make the subsequent "CNN degrades at higher resolution" finding meaningless  it would be a product of our selection, not of the architecture
# Tie-break: if the top configs are within noise of each other (which they are here, given only 10 validation trajectories), prefer the smaller model to reduce overfitting risk.
best = min(results, key=lambda r: r['val'])
print(f"\nBest config: {best}")